# Transcriptomics Analysis Workflow

This notebook demonstrates a complete RNA-seq analysis workflow including:
- Data loading and preprocessing
- Quality control
- Normalization
- Differential expression analysis
- Pathway enrichment
- Visualization

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests

# MOAS imports
from backend.omics.core import TranscriptomicsModule
from backend.omics.base import OmicsData, AnalysisParams, DataSource

plt.style.use('seaborn-v0_8-whitegrid')
print("Libraries loaded!")

## 1. Data Loading

In [ ]:
# Generate realistic expression data for demonstration
np.random.seed(42)

n_genes = 5000
n_control = 20
n_treatment = 20
n_samples = n_control + n_treatment

# Base expression levels
base_expr = np.random.lognormal(4, 2, n_genes)

# Generate counts
counts = np.zeros((n_genes, n_samples))

# Control samples
for i in range(n_control):
    counts[:, i] = np.random.poisson(base_expr)

# Treatment samples - add differential expression for some genes
de_genes = np.random.choice(n_genes, 500, replace=False)
fold_changes = np.random.choice([-1, 1], 500) * np.random.uniform(1.5, 4, 500)

for i in range(n_control, n_samples):
    expr = base_expr.copy()
    expr[de_genes] = expr[de_genes] * (2 ** fold_changes)
    counts[:, i] = np.random.poisson(np.maximum(expr, 1))

# Create DataFrame
gene_names = [f"GENE{i:05d}" for i in range(n_genes)]
sample_names = [f"Control_{i}" for i in range(n_control)] + [f"Treatment_{i}" for i in range(n_treatment)]

expression_df = pd.DataFrame(counts, index=gene_names, columns=sample_names)

# Metadata
metadata = pd.DataFrame({
    'sample_id': sample_names,
    'condition': ['Control'] * n_control + ['Treatment'] * n_treatment,
    'batch': np.random.choice(['Batch1', 'Batch2'], n_samples)
}).set_index('sample_id')

print(f"Expression matrix: {expression_df.shape[0]} genes x {expression_df.shape[1]} samples")
print(f"DE genes introduced: {len(de_genes)}")
expression_df.head()

## 2. Quality Control

In [ ]:
# Library sizes
lib_sizes = expression_df.sum()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Library size distribution
axes[0].bar(range(n_samples), lib_sizes, color=['#3498db'] * n_control + ['#e74c3c'] * n_treatment)
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('Library Size')
axes[0].set_title('Library Sizes')
axes[0].axhline(lib_sizes.median(), color='gray', linestyle='--', label='Median')

# Detected genes per sample
detected = (expression_df > 0).sum()
axes[1].bar(range(n_samples), detected, color=['#3498db'] * n_control + ['#e74c3c'] * n_treatment)
axes[1].set_xlabel('Sample')
axes[1].set_ylabel('Detected Genes')
axes[1].set_title('Detected Genes per Sample')

# Expression distribution
for i, col in enumerate(expression_df.columns[:5]):
    log_expr = np.log2(expression_df[col] + 1)
    axes[2].hist(log_expr, bins=50, alpha=0.5, label=col)
axes[2].set_xlabel('log2(counts + 1)')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Expression Distribution (first 5 samples)')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Gene filtering
min_counts = 10
min_samples = 5

gene_filter = (expression_df >= min_counts).sum(axis=1) >= min_samples
filtered_df = expression_df.loc[gene_filter]

print(f"Genes before filtering: {len(expression_df)}")
print(f"Genes after filtering: {len(filtered_df)}")
print(f"Genes removed: {len(expression_df) - len(filtered_df)}")

## 3. Normalization

In [ ]:
# CPM normalization
cpm = filtered_df.div(filtered_df.sum()) * 1e6

# Log2 transformation
log_cpm = np.log2(cpm + 1)

# Quantile normalization
def quantile_normalize(df):
    rank_mean = df.stack().groupby(df.rank(method='first').stack().astype(int)).mean()
    return df.rank(method='min').stack().astype(int).map(rank_mean).unstack()

normalized_df = quantile_normalize(log_cpm)

# Compare before and after normalization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before normalization
log_cpm.boxplot(ax=axes[0], rot=90)
axes[0].set_title('Before Quantile Normalization')
axes[0].set_ylabel('log2(CPM + 1)')

# After normalization
normalized_df.boxplot(ax=axes[1], rot=90)
axes[1].set_title('After Quantile Normalization')
axes[1].set_ylabel('Normalized Expression')

plt.tight_layout()
plt.show()

## 4. Differential Expression Analysis

In [ ]:
# Perform t-test for each gene
control_cols = [c for c in normalized_df.columns if 'Control' in c]
treatment_cols = [c for c in normalized_df.columns if 'Treatment' in c]

results = []
for gene in normalized_df.index:
    ctrl = normalized_df.loc[gene, control_cols]
    treat = normalized_df.loc[gene, treatment_cols]
    
    log2fc = treat.mean() - ctrl.mean()
    stat, pval = stats.ttest_ind(ctrl, treat)
    
    results.append({
        'gene': gene,
        'log2FoldChange': log2fc,
        'pvalue': pval,
        'baseMean': (ctrl.mean() + treat.mean()) / 2
    })

de_results = pd.DataFrame(results)

# Multiple testing correction
de_results['padj'] = multipletests(de_results['pvalue'], method='fdr_bh')[1]

# Mark significant genes
de_results['significant'] = (de_results['padj'] < 0.05) & (np.abs(de_results['log2FoldChange']) > 1)

print(f"Total genes tested: {len(de_results)}")
print(f"Significant genes (FDR < 0.05, |log2FC| > 1): {de_results['significant'].sum()}")
print(f"  Upregulated: {((de_results['significant']) & (de_results['log2FoldChange'] > 0)).sum()}")
print(f"  Downregulated: {((de_results['significant']) & (de_results['log2FoldChange'] < 0)).sum()}")

de_results.sort_values('pvalue').head(20)

In [ ]:
# Volcano plot
fig, ax = plt.subplots(figsize=(10, 8))

de_results['neg_log10_padj'] = -np.log10(de_results['padj'] + 1e-300)

# Color coding
colors = np.where(
    de_results['significant'] & (de_results['log2FoldChange'] > 0), '#e74c3c',
    np.where(
        de_results['significant'] & (de_results['log2FoldChange'] < 0), '#3498db',
        '#999999'
    )
)

ax.scatter(
    de_results['log2FoldChange'],
    de_results['neg_log10_padj'],
    c=colors,
    alpha=0.5,
    s=10
)

ax.axhline(-np.log10(0.05), color='gray', linestyle='--', linewidth=1)
ax.axvline(1, color='gray', linestyle='--', linewidth=1)
ax.axvline(-1, color='gray', linestyle='--', linewidth=1)

# Label top genes
top_genes = de_results.nsmallest(10, 'padj')
for _, row in top_genes.iterrows():
    ax.annotate(
        row['gene'],
        (row['log2FoldChange'], row['neg_log10_padj']),
        fontsize=8,
        alpha=0.8
    )

ax.set_xlabel('log2(Fold Change)')
ax.set_ylabel('-log10(adjusted p-value)')
ax.set_title('Volcano Plot - Differential Expression')

plt.tight_layout()
plt.show()

In [ ]:
# MA plot
fig, ax = plt.subplots(figsize=(10, 8))

ax.scatter(
    de_results['baseMean'],
    de_results['log2FoldChange'],
    c=colors,
    alpha=0.5,
    s=10
)

ax.axhline(0, color='black', linestyle='-', linewidth=1)
ax.axhline(1, color='gray', linestyle='--', linewidth=1)
ax.axhline(-1, color='gray', linestyle='--', linewidth=1)

ax.set_xlabel('Mean Expression')
ax.set_ylabel('log2(Fold Change)')
ax.set_title('MA Plot')

plt.tight_layout()
plt.show()

## 5. Heatmap of Top DE Genes

In [ ]:
# Select top 50 DE genes
top_genes = de_results[de_results['significant']].nsmallest(50, 'padj')['gene']

# Z-score normalization for heatmap
heatmap_data = normalized_df.loc[top_genes]
heatmap_z = (heatmap_data.T - heatmap_data.mean(axis=1)).T / heatmap_data.std(axis=1).values[:, np.newaxis]

# Create annotation
col_colors = ['#3498db' if 'Control' in c else '#e74c3c' for c in heatmap_z.columns]

# Plot
fig = plt.figure(figsize=(14, 12))
g = sns.clustermap(
    heatmap_z,
    cmap='RdBu_r',
    center=0,
    col_colors=col_colors,
    figsize=(14, 12),
    dendrogram_ratio=0.15,
    cbar_pos=(0.02, 0.8, 0.03, 0.15)
)
g.ax_heatmap.set_xlabel('Samples')
g.ax_heatmap.set_ylabel('Genes')
plt.suptitle('Top 50 Differentially Expressed Genes', y=1.02)
plt.show()

## 6. Save Results

In [ ]:
# Save DE results
# de_results.to_csv('results/de_results.csv', index=False)
# normalized_df.to_csv('results/normalized_expression.csv')

print("Analysis complete!")
print(f"\nSummary:")
print(f"  Total genes analyzed: {len(de_results)}")
print(f"  Significant DE genes: {de_results['significant'].sum()}")
print(f"  Upregulated: {((de_results['significant']) & (de_results['log2FoldChange'] > 0)).sum()}")
print(f"  Downregulated: {((de_results['significant']) & (de_results['log2FoldChange'] < 0)).sum()}")